## Model Building (2023+ Data & Expanded Features - RF/LGBM Focus)
- Fetch/Load data from 2023 season onwards.
- Feature Engineering: Use player_name, pitch_type, release_speed, zone (for norm_zone), stand, p_throws, balls, strikes, pfx_x, pfx_z.
- Preprocessing: Handle NaNs, One-Hot Encode categoricals, Scale numericals.
- Model Experiments: Random Forest, LightGBM.
- Evaluate using PR AUC, F1.
- Save best model, scaler, and df_shell.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger().setLevel(logging.ERROR)

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
    auc
)
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import numpy as np
from pybaseball import statcast
import os
import time


%matplotlib inline
%load_ext autoreload
%autoreload 2


In [3]:
section_start_time = time.time()

In [4]:
# Import LightGBM safely
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print("LightGBM is available.")
except ImportError:
    print("⚠️ LightGBM not installed. Skipping Gradient Boosting experiment.")
    LGBM_AVAILABLE = False

LightGBM is available.


In [5]:
# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Don’t truncate column content
pd.set_option('display.max_colwidth', None)

# Optional: widen the display
pd.set_option('display.width', 0)


In [6]:
pl.Config.set_fmt_str_lengths(900)
pl.Config.set_tbl_width_chars(900)
pl.Config(tbl_rows=-1)

In [7]:
START_DATE = '2023-03-30'
# Use a slightly earlier end date if running mid-season to avoid incomplete data days
# END_DATE = (datetime.date.today() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')
END_DATE = '2024-9-29' # Or use a fixed date

In [8]:
# Define file paths (adjust if your notebook isn't in the 'model' folder)
# Assumes notebook is in 'model' folder, and data fixtures are in 'backend/mapping_heat/fixtures'
try:
    # Try navigating up one level from 'model' folder
    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..')) 
except NameError:
     # Fallback if __file__ is not defined (e.g., interactive session)
     base_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) 
     
CSV_FILENAME = f"pitching_data_{START_DATE}_to_{END_DATE}.csv"
CSV_PATH = os.path.join(base_dir, "backend", "mapping_heat", "fixtures", CSV_FILENAME)

MODEL_DIR = "./" # Directory to save model outputs in the current working dir
MODEL_FILENAME = os.path.join(MODEL_DIR, f"pitch_prob_model_2023plus_tree_expanded_feat_{END_DATE}.sav")
SCALER_FILENAME = os.path.join(MODEL_DIR, f"pitch_scaler_2023plus_tree_expanded_feat_{END_DATE}.sav")
SHELL_FILENAME = os.path.join(MODEL_DIR, f"df_shell_2023plus_tree_expanded_feat_{END_DATE}.csv")
SPLIT_DATA_FILENAME = os.path.join(MODEL_DIR, f"train_test_split_data_{START_DATE}_to_{END_DATE}.npz")

In [9]:
# --- Control Flags ---
LOAD_SAVED_SPLIT = True # <<< SET TO True TO TRY LOADING SAVED DATA >>>

--- Feature Lists ---

In [10]:
numerical_features = ['release_speed', 'balls', 'strikes', 'pfx_x', 'pfx_z']
# Categoricals used for final encoding (after norm_zone created)
categorical_features_final = ['player_name', 'pitch_type', 'norm_zone', 'stand', 'p_throws']
# Columns needed from source data
cols_to_keep = [
    'player_name', 'pitch_type', 'release_speed', 'zone', 'stand', 'p_throws',
    'balls', 'strikes', 'pfx_x', 'pfx_z', 'description', 'game_date' # Keep game_date temporarily
]
# Categoricals needed for initial imputation
categorical_features_initial = ['player_name', 'pitch_type', 'zone', 'stand', 'p_throws']
target_source_col = 'description'

In [11]:
# Initialize variables that might be loaded or created
X_train_np, X_test_np, y_train, y_test = None, None, None, None
scaler = None
final_feature_names = None
data_ready_for_scaling = False # Flag

In [12]:
statcast??

Signature:
statcast(
    start_dt: str = None,
    end_dt: str = None,
    team: str = None,
    verbose: bool = True,
    parallel: bool = True,
) -> pandas.core.frame.DataFrame
Source:   
def statcast(start_dt: str = None, end_dt: str = None, team: str = None,
             verbose: bool = True, parallel: bool = True) -> pd.DataFrame:
    """
    Pulls statcast play-level data from Baseball Savant for a given date range.

    INPUTS:
    start_dt: YYYY-MM-DD : the first date for which you want statcast data
    end_dt: YYYY-MM-DD : the last date for which you want statcast data
    team: optional (defaults to None) : city abbreviation of the team you want data for (e.g. SEA or BOS)
    verbose: bool (defaults to True) : whether to print updates on query progress
    parallel: bool (defaults to True) : whether to parallelize HTTP requests in large queries

    If no arguments are provided, this will return yesterday's statcast data.
    If one date is provided, it will return that date

### Optional: Load Pre-Split Data & Scaler

In [13]:
start_load_split_time = time.time()
if LOAD_SAVED_SPLIT:
    print("Attempting to load saved data...")
    # Check if BOTH files exist before trying to load
    if os.path.exists(SPLIT_DATA_FILENAME) and os.path.exists(SCALER_FILENAME):
        print(f"📂 Found pre-split data: {SPLIT_DATA_FILENAME}")
        print(f"📂 Found saved scaler: {SCALER_FILENAME}")
        try:
            print("Loading arrays from .npz file...")
            with np.load(SPLIT_DATA_FILENAME, allow_pickle=True) as data:
                X_train_np = data['X_train'] # Load unscaled arrays
                X_test_np = data['X_test']
                y_train = data['y_train']
                y_test = data['y_test']
                final_feature_names = data['feature_names'].tolist()
            print(f"✅ Loaded arrays (Train shape: {X_train_np.shape}, Test shape: {X_test_np.shape}).")

            print("Loading scaler from .sav file...")
            with open(SCALER_FILENAME, 'rb') as f:
                  scaler = pickle.load(f)
            print("✅ Loaded saved scaler.")

            # Verify all necessary components were loaded
            if X_train_np is not None and X_test_np is not None and y_train is not None and y_test is not None and scaler is not None and final_feature_names is not None:
                data_is_loaded_and_split = True # Data and scaler loaded successfully
                print("Data and scaler ready for scaling application.")
                print(f"⏱️ Loading split data and scaler took: {time.time() - start_load_split_time:.2f} seconds")
            else:
                 print("❌ Loading seemed complete, but some variables are still None. Proceeding with full processing.")
                 data_is_loaded_and_split = False

        except Exception as e:
            print(f"❌ Error loading pre-split data or scaler: {e}. Proceeding with full data load and preprocessing.")
            data_is_loaded_and_split = False
    else:
        print(f"⚠️ Required files not found (Split data: {os.path.exists(SPLIT_DATA_FILENAME)}, Scaler: {os.path.exists(SCALER_FILENAME)}). Proceeding with full data load and preprocessing.")
        data_is_loaded_and_split = False
else:
    print("LOAD_SAVED_SPLIT is False. Proceeding with full data load and preprocessing.")
    data_is_loaded_and_split = False

Attempting to load saved data...
📂 Found pre-split data: ./train_test_split_data_2023-03-30_to_2024-9-29.npz
📂 Found saved scaler: ./pitch_scaler_2023plus_tree_expanded_feat_2024-9-29.sav
Loading arrays from .npz file...
✅ Loaded arrays (Train shape: (1031886, 1427), Test shape: (442238, 1427)).
Loading scaler from .sav file...
✅ Loaded saved scaler.
Data and scaler ready for scaling application.
⏱️ Loading split data and scaler took: 4.72 seconds


In [14]:
load_df = True

### Data collection

In [15]:
df = None # Use df for the INITIAL Polars DataFrame

if not data_ready_for_scaling or load_df: # If data wasn't successfully loaded from saved split
    section_start_time = time.time()
    print(f"\n--- 1. Data Loading/Fetching ---")
    print(f"Looking for CSV at: {CSV_PATH}") # Debugging print
    fetch_new = False # Default to NOT fetching unless necessary

    if os.path.exists(CSV_PATH):
        print(f"📂 Found existing file: {CSV_PATH}")
        try:
            df = pl.read_csv(CSV_PATH, try_parse_dates=True, ignore_errors=True)
            print(f"Loaded {df.height} rows from CSV.")

            if df.height > 0 and 'game_date' in df.columns:
                # Attempt date parsing robustly
                if df['game_date'].dtype != pl.Date:
                    print("Parsing 'game_date' column...")
                    # Keep original date for comparison if parsing fails
                    df = df.with_columns(
                        pl.col('game_date').str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias('game_date_parsed')
                    )
                    # Check how many rows failed parsing
                    parse_failures = df.filter(pl.col('game_date_parsed').is_null()).height
                    if parse_failures > 0:
                         print(f"⚠️ Warning: {parse_failures} rows had unparseable 'game_date'.")
                    # Keep only successfully parsed rows for date range check
                    df_parsed_dates = df.filter(pl.col('game_date_parsed').is_not_null())
                    # Decide whether to drop original 'game_date' or keep it
                    # df = df.drop('game_date').rename({'game_date_parsed': 'game_date'}) # Option 1: replace
                    df = df.drop('game_date_parsed') # Option 2: Keep original (might be safer if parsing fails)


                # Check date range using the original or parsed date column
                date_col_to_check = 'game_date_parsed' if 'game_date_parsed' in df_parsed_dates.columns else 'game_date'

                if date_col_to_check in df_parsed_dates.columns and df_parsed_dates[date_col_to_check].dtype == pl.Date:
                    min_date = df_parsed_dates.select(pl.col(date_col_to_check).min()).item()
                    max_date = df_parsed_dates.select(pl.col(date_col_to_check).max()).item()

                    if min_date is not None and max_date is not None:
                        print(f"Data in CSV ranges from {min_date} to {max_date}")
                        start_dt_obj = datetime.datetime.strptime(START_DATE, '%Y-%m-%d').date()
                        end_dt_obj = datetime.datetime.strptime(END_DATE, '%Y-%m-%d').date()

                        # Check start date and if end date is reasonably recent
                        if min_date <= start_dt_obj and max_date >= (end_dt_obj - datetime.timedelta(days=7)):
                            print("✅ Existing CSV covers the required date range and is recent enough.")
                            fetch_new = False
                        else:
                            print(f"⚠️ CSV data range issue (Needs: {START_DATE} to {END_DATE}). Fetching fresh data.")
                            fetch_new = True
                    else:
                        print("⚠️ Could not determine date range from CSV. Fetching fresh data.")
                        fetch_new = True
                else:
                    print(f"⚠️ '{date_col_to_check}' column not found or not Date type after parsing attempt. Fetching fresh data.")
                    fetch_new = True # Force fetch if date checks fail
            else:
                 print(f"⚠️ CSV file is empty or missing 'game_date' column. Fetching fresh data.")
                 fetch_new = True # Force fetch if file empty
        except Exception as e:
            print(f"❌ Error reading/validating CSV: {e}. Fetching fresh data.")
            df = None # Ensure df is reset if reading failed
            fetch_new = True
    else:
        print(f" File {CSV_PATH} not found.")
        fetch_new = True

    # --- Fetching Logic ---
    if fetch_new:
        print(f"🌐 Fetching data with pybaseball from {START_DATE} to {END_DATE}...")
        try:
            df_pd_fetched = statcast(start_dt=START_DATE, end_dt=END_DATE, verbose=True, parallel=True)

            if df_pd_fetched is not None and not df_pd_fetched.empty:
                 print("Converting fetched pandas data to Polars...")
                 df = pl.from_pandas(df_pd_fetched) # Assign to main Polars df variable
                 del df_pd_fetched # Clean up memory

                 print("Ensuring save directory exists...")
                 # Make sure the directory exists before trying to save
                 try:
                     os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
                     print(f"Saving fetched data ({df.height} rows) to {CSV_PATH}...")
                     df.write_csv(CSV_PATH)
                     print("💾 Saved successfully.")
                 except Exception as save_err:
                      print(f"❌ Error saving fetched data to {CSV_PATH}: {save_err}")
                      print("   Proceeding with fetched data in memory, but it won't be saved.")

            else:
                 print("❌ No data fetched from pybaseball. statcast() returned empty or None.")
                 # Decide how to handle: exit or try to use old CSV if it exists?
                 if df is not None:
                      print("⚠️ Using potentially outdated/invalid data from CSV as fallback.")
                 else:
                      print("❌ No data available. Exiting.")
                      exit() # Exit if fetch fails and no prior df was loaded

        except Exception as e:
            print(f"❌ Error during pybaseball fetch or conversion: {e}.")
            if df is not None:
                print("⚠️ Using potentially outdated/invalid data from CSV as fallback.")
            else:
                print("❌ No data available. Exiting.")
                exit() # Exit if fetch fails and no prior df

    # --- Final Check ---
    if df is None or df.height == 0:
        print("❌ ERROR: DataFrame 'df' is empty or None after loading/fetching attempt. Cannot proceed.")
        exit() # Stop execution if no data

    print(f"Final loaded/fetched data shape: {df.shape}")
    print(f"⏱️ Data Loading/Fetching Section took: {time.time() - section_start_time:.2f} seconds")


--- 1. Data Loading/Fetching ---
Looking for CSV at: /Users/brandonlyons/Documents/MappingHeat/backend/mapping_heat/fixtures/pitching_data_2023-03-30_to_2024-9-29.csv
📂 Found existing file: /Users/brandonlyons/Documents/MappingHeat/backend/mapping_heat/fixtures/pitching_data_2023-03-30_to_2024-9-29.csv
Loaded 1474124 rows from CSV.
Parsing 'game_date' column...
❌ Error reading/validating CSV: invalid series dtype: expected `String`, got `datetime[μs]` for series with name `game_date`. Fetching fresh data.
🌐 Fetching data with pybaseball from 2023-03-30 to 2024-9-29...
This is a large query, it may take a moment to complete
Skipping offseason dates


100%|█████████████████████████████████████████████████████████████████████████████████████████| 430/430 [01:15<00:00,  5.71it/s]


Converting fetched pandas data to Polars...
Ensuring save directory exists...
Saving fetched data (1474124 rows) to /Users/brandonlyons/Documents/MappingHeat/backend/mapping_heat/fixtures/pitching_data_2023-03-30_to_2024-9-29.csv...
💾 Saved successfully.
Final loaded/fetched data shape: (1474124, 113)
⏱️ Data Loading/Fetching Section took: 87.69 seconds


In [16]:
df.columns

['pitch_type',
 'game_date',
 'release_speed',
 'release_pos_x',
 'release_pos_z',
 'player_name',
 'batter',
 'pitcher',
 'events',
 'description',
 'spin_dir',
 'spin_rate_deprecated',
 'break_angle_deprecated',
 'break_length_deprecated',
 'zone',
 'des',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'hit_location',
 'bb_type',
 'balls',
 'strikes',
 'game_year',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'on_3b',
 'on_2b',
 'on_1b',
 'outs_when_up',
 'inning',
 'inning_topbot',
 'hc_x',
 'hc_y',
 'tfs_deprecated',
 'tfs_zulu_deprecated',
 'umpire',
 'sv_id',
 'vx0',
 'vy0',
 'vz0',
 'ax',
 'ay',
 'az',
 'sz_top',
 'sz_bot',
 'hit_distance_sc',
 'launch_speed',
 'launch_angle',
 'effective_speed',
 'release_spin_rate',
 'release_extension',
 'game_pk',
 'fielder_2',
 'fielder_3',
 'fielder_4',
 'fielder_5',
 'fielder_6',
 'fielder_7',
 'fielder_8',
 'fielder_9',
 'release_pos_y',
 'estimated_ba_using_speedangle',
 'estimated_woba_using_speedangle',
 'w

In [17]:
pd.set_option('display.max_columns', None)
df.head()

pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,…,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle
str,datetime[ns],f64,f64,f64,str,i64,i64,str,str,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,i64,str,i64,i64,i64,f64,f64,f64,f64,i64,i64,i64,i64,i64,str,…,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64
"""FF""",2024-09-29 00:00:00,94.8,-1.08,6.25,"""Nelson, Ryne""",444482,669194,"""field_out""","""hit_into_play""",null,null,null,null,14,"""David Peralta grounds out, first baseman Christian Walker to pitcher Ryne Nelson.""","""R""","""L""","""R""","""AZ""","""SD""","""X""",3,"""ground_ball""",0,0,2024,-0.33,1.61,0.16,1.5,null,null,null,2,9,"""Top""",…,"""4-Seam Fastball""",11,2,2,11,2,11,2,11,"""Strategic""","""Standard""",194,0.0,-0.256,80.7,7.6,0.169,0.256,98.1,9,-9,1.0,0.0,26,36,26,37,1,3,21,1,null,6,0.93,0.33,-0.33,57.3
"""FC""",2024-09-29 00:00:00,90.1,-1.59,6.04,"""Nelson, Ryne""",630105,669194,"""field_out""","""hit_into_play""",null,null,null,null,7,"""Jake Cronenworth lines out to third baseman Eugenio Suárez.""","""R""","""L""","""R""","""AZ""","""SD""","""X""",5,"""line_drive""",1,1,2024,0.22,0.65,-0.48,1.79,null,null,null,1,9,"""Top""",…,"""Cutter""",11,2,2,11,2,11,2,11,"""Infield shade""","""Standard""",181,0.0,-0.243,68.9,7.5,0.808,0.243,102.1,9,-9,1.0,0.0,26,30,26,30,1,3,21,2,null,2,2.15,-0.22,0.22,48.9
"""FF""",2024-09-29 00:00:00,94.2,-1.24,6.14,"""Nelson, Ryne""",630105,669194,null,"""foul""",null,null,null,null,12,"""Jake Cronenworth lines out to third baseman Eugenio Suárez.""","""R""","""L""","""R""","""AZ""","""SD""","""S""",null,null,1,0,2024,-0.41,1.66,0.17,3.48,null,null,null,1,9,"""Top""",…,"""4-Seam Fastball""",11,2,2,11,2,11,2,11,"""Infield shade""","""Standard""",199,0.0,-0.051,63.3,5.8,null,0.051,88.0,9,-9,1.0,0.0,26,30,26,30,1,3,21,2,null,2,0.91,0.41,-0.41,52.3
"""FF""",2024-09-29 00:00:00,95.2,-0.93,6.23,"""Nelson, Ryne""",630105,669194,null,"""ball""",null,null,null,null,14,"""Jake Cronenworth lines out to third baseman Eugenio Suárez.""","""R""","""L""","""R""","""AZ""","""SD""","""B""",null,null,0,0,2024,-0.22,1.65,1.52,1.21,null,null,null,1,9,"""Top""",…,"""4-Seam Fastball""",11,2,2,11,2,11,2,11,"""Infield shade""","""Standard""",190,0.0,0.039,null,null,null,-0.039,null,9,-9,1.0,0.0,26,30,26,30,1,3,21,2,null,2,0.88,0.22,-0.22,57.0
"""FF""",2024-09-29 00:00:00,95.0,-1.06,6.21,"""Nelson, Ryne""",665487,669194,"""field_out""","""hit_into_play""",null,null,null,null,4,"""Fernando Tatis Jr. flies out sharply to left fielder Jake McCarthy.""","""R""","""R""","""R""","""AZ""","""SD""","""X""",7,"""fly_ball""",0,0,2024,-0.31,1.61,-0.71,2.84,null,null,null,0,9,"""Top""",…,"""4-Seam Fastball""",11,2,2,11,2,11,2,11,"""Standard""","""Standard""",189,0.0,-0.254,82.3,7.9,0.048,0.254,100.9,9,-9,1.0,0.0,26,25,26,25,1,3,21,2,null,2,0.9,0.31,0.31,54.9


In [18]:
df.shape

(1474124, 113)

In [19]:
type(df)

polars.dataframe.frame.DataFrame

### Find features to use for model

#### Pitchers

In [20]:
pitchers_pl = (
    df.group_by("player_name") # Group by the player name
    .agg(
        pl.count().alias("count") # Count occurrences in each group and name the new column 'count'
    )
    .rename({"player_name": "pitcher"}) # Rename the player name column to 'pitcher'
    .sort("count", descending=True) # Sort by count descending (value_counts implicitly sorts)
)

pitchers_pl.head()

pitcher,count
str,u32
"""Wheeler, Zack""",6817
"""Nola, Aaron""",6699
"""Cease, Dylan""",6525
"""Webb, Logan""",6455
"""Gallen, Zac""",6440


In [21]:
pitchers_pl['count'].mean()

1063.5815295815296

In [22]:
pitchers_pl['count'].min()

1

In [23]:
pitchers_pl['count'].describe()

statistic,value
str,f64
"""count""",1386.0
"""null_count""",0.0
"""mean""",1063.58153
"""std""",1400.900465
"""min""",1.0
"""25%""",42.0
"""50%""",472.0
"""75%""",1591.0
"""max""",6817.0


#### Release speed

In [24]:
df.group_by("release_speed").count().sort("count", descending=True).head(10)                     

release_speed,count
f64,u32
null,15160
93.9,11476
93.5,11380
94.0,11331
93.7,11313
94.1,11311
93.8,11302
94.2,11291
94.3,11281


In [28]:
# rl_speed = df.release_speed.value_counts().to_frame().reset_index().rename(columns={'index': 'speed', 'release_speed': 'count'})

rl_speed = (
    df.group_by("release_speed")
    .agg(
        pl.count().alias("count")
    )
    .sort("count", descending=True)) # Sort by count descending (value_counts implicitly sorts)

rl_speed.head()

release_speed,count
f64,u32
null,15160
93.9,11476
93.5,11380
94.0,11331
93.7,11313


In [26]:
rl_speed['release_speed'].describe()

statistic,value
str,f64
"""count""",706.0
"""null_count""",1.0
"""mean""",69.721671
"""std""",20.447082
"""min""",31.9
"""25%""",52.1
"""50%""",69.8
"""75%""",87.4
"""max""",105.5


In [29]:
rl_speed.filter(
    pl.col("release_speed").is_null()
)

release_speed,count
f64,u32
null,15160


#### Pitch outcome

In [30]:
df['type'].value_counts()

type,count
str,u32
"""X""",258190
"""B""",527167
"""S""",688767


In [31]:
print("\nPitch Outcome ('type') Counts:")
pitch_outcome_counts = df.group_by("type").agg(pl.count()).sort("count", descending=True)
print(pitch_outcome_counts)

# Calculate percentages (example for 'S')
total_pitches = df.height
strike_count = pitch_outcome_counts.filter(pl.col('type') == 'S').select('count').item()
ball_count = pitch_outcome_counts.filter(pl.col('type') == 'B').select('count').item()
in_play_count = pitch_outcome_counts.filter(pl.col('type') == 'X').select('count').item()

if total_pitches > 0:
    print(f"Strike %: {(strike_count / total_pitches) * 100:.2f}%")
    print(f"Ball %: {(ball_count / total_pitches) * 100:.2f}%")
    print(f"In Play %: {(in_play_count / total_pitches) * 100:.2f}%")



Pitch Outcome ('type') Counts:
shape: (3, 2)
┌──────┬────────┐
│ type ┆ count  │
│ ---  ┆ ---    │
│ str  ┆ u32    │
╞══════╪════════╡
│ S    ┆ 688767 │
│ B    ┆ 527167 │
│ X    ┆ 258190 │
└──────┴────────┘
Strike %: 46.72%
Ball %: 35.76%
In Play %: 17.51%


#### Outcome per at bat

In [32]:
print("\nAt-Bat Event Counts:")
event_counts = df.group_by("events").agg(pl.count()).sort("count", descending=True)
event_counts.head(20)


At-Bat Event Counts:


events,count
str,u32
null,1093307
"""field_out""",151927
"""strikeout""",86330
"""single""",54064
"""walk""",30953
"""double""",16677
"""home_run""",11789
"""force_out""",7342
"""grounded_into_double_play""",6948


#### Description per pitch outcome

In [33]:
print("\nPitch Description Counts:")
description_counts = df.group_by("description").agg(pl.count()).sort("count", descending=True)
description_counts.head(20) 


Pitch Description Counts:


description,count
str,u32
"""ball""",491099
"""foul""",263313
"""hit_into_play""",258208
"""called_strike""",241392
"""swinging_strike""",157823
"""blocked_ball""",31671
"""foul_tip""",14995
"""swinging_strike_blocked""",8512
"""hit_by_pitch""",4295


###### Pitch zones

In [34]:
df.filter(pl.col("zone").is_null()).select("description").head()

description
str
"""foul"""
"""foul"""
"""ball"""
"""swinging_strike"""
"""swinging_strike"""


In [35]:
print("\nPitch Zone Counts (Top 15):")
zone_counts = df.group_by("zone").agg(pl.count()).sort("count", descending=True)
zone_counts.head(15)


Pitch Zone Counts (Top 15):


zone,count
i64,u32
14,273058
13,182522
11,161695
12,122063
5,108970
8,92798
6,87757
4,86496
9,81632


##### Pitch legend

- AB Automatic Ball
- AS Automatic Strike
- CH Change-​up
- CU Curveball
- EP Eephus
- FC Cutter
- FF Four-Seam Fastball
- FO Forkball
- FS Splitter
- FT Two-Seam Fastball (synonymous with SI)
- GY Gyroball
- IN Intentional Ball
- KC Knuckle Curve
- KN Knuckleball
- NP No Pitch
- PO Pitchout
- SC Screwball
- SI Sinker (synonymous with FT)
- SL Slider
- UN Unknown

In [36]:
print("\nPitch Type Counts:")
pitch_type_counts = df.group_by("pitch_type").agg(pl.count()).sort("count", descending=True)
pitch_type_counts


Pitch Type Counts:


pitch_type,count
str,u32
"""FF""",467560
"""SL""",233152
"""SI""",227994
"""CH""",153747
"""FC""",117614
"""CU""",95128
"""ST""",87441
"""FS""",38957
"""KC""",27802


In [37]:
print("\nPitch Type / Pitch Name Counts:")
pitch_name_combo_counts = (
    df.group_by(['pitch_type', 'pitch_name'])
    .agg(pl.count())
    .sort('count', descending=True)
)
pitch_name_combo_counts


Pitch Type / Pitch Name Counts:


pitch_type,pitch_name,count
str,str,u32
"""FF""","""4-Seam Fastball""",467560
"""SL""","""Slider""",233152
"""SI""","""Sinker""",227994
"""CH""","""Changeup""",153747
"""FC""","""Cutter""",117614
"""CU""","""Curveball""",95128
"""ST""","""Sweeper""",87441
"""FS""","""Split-Finger""",38957
"""KC""","""Knuckle Curve""",27802


In [70]:
if not LOAD_SAVED_SPLIT or not load_df:
    print("yues")

yues


In [60]:
LOAD_SAVED_SPLIT

True

### Feature Selection & Engineering


In [66]:
if not LOAD_SAVED_SPLIT or load_df:
    section_start_time = time.time()
    print("\n--- 2. Feature Selection & Engineering ---")

    # Define columns to keep + those needed for engineering target/norm_zone
    cols_to_keep = [
        'player_name', 'pitch_type', 'release_speed', 'zone', 'stand', 'p_throws',
        'balls', 'strikes', 'pfx_x', 'pfx_z', 'description'
    ]
    # numerical_features and categorical_features_initial defined in config section
    target_source_col = 'description'

    # Check for column existence in the loaded Polars DataFrame `df`
    missing_cols = [col for col in cols_to_keep if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}. Cannot proceed.")
        exit()
    else:
        print(f"Selecting columns: {cols_to_keep}")
        df_selected = df.select(cols_to_keep) # Use df here

    print(f"Shape after column selection: {df_selected.shape}")

    # Clone for safe modification
    df_processed = df_selected.clone() # Use df_processed for intermediate steps
    print(f"⏱️ Feature Selection took: {time.time() - section_start_time:.2f} seconds")



--- 2. Feature Selection & Engineering ---
Selecting columns: ['player_name', 'pitch_type', 'release_speed', 'zone', 'stand', 'p_throws', 'balls', 'strikes', 'pfx_x', 'pfx_z', 'description']
Shape after column selection: (1474124, 11)
⏱️ Feature Selection took: 0.00 seconds


### Preprocessing

In [65]:
if not LOAD_SAVED_SPLIT or load_df:
    section_start_time = time.time()
    print("\n--- 3. Preprocessing ---")

    # Handle Missing Values (using Polars on df_pl_processed)
    print("Handling missing values...")
    imputation_start_time = time.time()
    categorical_features_initial = ['player_name', 'pitch_type', 'zone', 'stand', 'p_throws']

    # Impute numerical with median
    for col in numerical_features:
        if str(df_pl_processed[col].dtype) in ['Float32', 'Float64', 'Int8', 'Int16', 'Int32', 'Int64', 'UInt8', 'UInt16', 'UInt32', 'UInt64']:
            median_val = df_pl_processed.select(pl.col(col).median().cast(pl.Float64)).item()
            if median_val is not None:
                null_count = df_pl_processed[col].is_null().sum()
                if null_count > 0:
                     #print(f"Imputing {null_count} NaNs in '{col}' with median: {median_val:.2f}")
                     df_pl_processed = df_pl_processed.with_columns(
                         pl.col(col).fill_null(median_val).cast(pl.Float64)
                     )
            else:
                 print(f"⚠️ Column '{col}' is numeric but median is None. Filling NaNs with 0.0.")
                 df_pl_processed = df_pl_processed.with_columns(pl.col(col).fill_null(0.0).cast(pl.Float64))
        else:
             print(f"⚠️ Column '{col}' expected numeric, has {df_pl_processed[col].dtype}. Skipping.")

    # Impute categorical (including 'zone' before conversion) with 'Unknown'
    for col in categorical_features_initial:
         if col in df_pl_processed.columns:
              if df_pl_processed[col].dtype != pl.Utf8:
                   df_pl_processed = df_pl_processed.with_columns(pl.col(col).cast(pl.Utf8))
              null_count = df_pl_processed[col].is_null().sum()
              if null_count > 0:
                 #print(f"Imputing {null_count} NaNs in '{col}' with 'Unknown'")
                 df_pl_processed = df_pl_processed.with_columns(pl.col(col).fill_null('Unknown'))

    print(f"Imputation took: {time.time() - imputation_start_time:.2f}s")


--- 3. Preprocessing ---
Handling missing values...


NameError: name 'df_pl_processed' is not defined

In [52]:
# Create 'norm_zone'
print("Creating 'norm_zone' feature...")
df_pl_processed = df_pl_processed.with_columns(
    pl.col('zone').alias('norm_zone') # Zone is now Utf8 and imputed
)

Creating 'norm_zone' feature...


NameError: name 'df_pl_processed' is not defined

In [ ]:
# Create 'target' variable
print("Creating 'target' variable...")
df_pl_processed = df_pl_processed.with_columns(
    pl.when(pl.col(target_source_col).is_in(['hit_into_play_no_out', 'hit_into_play_score']))
    .then(pl.lit(1))
    .otherwise(pl.lit(0))
    .alias('target')
)

--- Select final features & Convert to Pandas for Encoding ---


In [ ]:
conversion_start_time = time.time()
# categorical_features_final defined in config section
feature_cols_final = numerical_features + categorical_features_final
print(f"\nSelecting final features: {feature_cols_final}")

X_pl = df_pl_processed.select(feature_cols_final) # Use X_pl for final Polars features
y_pl_s = df_pl_processed['target'] # Use y_pl_s for Polars target Series

In [ ]:
print(f"Converting features ({X_pl.shape}) to Pandas for encoding...")
X_pd_df = X_pl.to_pandas() # Use X_pd_df for intermediate Pandas DF
y_pd_series = y_pl_s.to_pandas() # Use y_pd_series for intermediate Pandas Series
del X_pl, y_pl_s, df_pl_processed # Free memory
print(f"Conversion to Pandas took: {time.time() - conversion_start_time:.2f}s")

--- Convert to Pandas/NumPy for Sklearn Workflow ---

In [ ]:
# One-Hot Encoding (using Pandas)
encoding_start_time = time.time()
print(f"\nOne-hot encoding categorical features: {categorical_features_final}")
# Ensure all categorical columns are treated as strings before encoding
for col in categorical_features_final:
    if col in X_pd_df.columns:
        X_pd_df[col] = X_pd_df[col].astype(str)

In [ ]:
X_pd_df_encoded = pd.get_dummies(X_pd_df, columns=categorical_features_final, dummy_na=False, dtype=np.int8) # Use smaller int dtype
del X_pd_df # Free memory
print(f"Shape after encoding: {X_pd_df_encoded.shape}")
print(f"Number of features after encoding: {len(X_pd_df_encoded.columns)}")
print(f"Encoding took: {time.time() - encoding_start_time:.2f}s")

In [ ]:
# Convert to NumPy for splitting, scaling, and modeling
conversion_np_start_time = time.time()
print("\nConverting encoded features and target to NumPy arrays...")
X_np_encoded = X_pd_df_encoded.to_numpy()
y_np = y_pd_series.to_numpy()
final_feature_names = X_pd_df_encoded.columns.tolist() # Get names *after* encoding
del X_pd_df_encoded, y_pd_series # Free memory
print(f"Conversion to NumPy took: {time.time() - conversion_np_start_time:.2f}s")

--- Data Splitting ---

In [ ]:
if not data_is_loaded_and_split:
    print("\nSplitting data into training and testing sets...")
    X_train_np, X_test_np, y_train, y_test = train_test_split(
        X_np_encoded,
        y_np,
        test_size=0.30,
        random_state=42,
        stratify=y_np # Important for imbalanced data
    )
    print(f"Training set shape: X={X_train_np.shape}, y={y_train.shape}")
    print(f"Test set shape: X={X_test_np.shape}, y={y_test.shape}")
    print(f"Positive class proportion in Train: {y_train.mean():.4f}, Test: {y_test.mean():.4f}")

In [ ]:
# --- Save Split Data ---
if not data_is_loaded_and_split:
    print("\n💾 Saving split data arrays to NPZ file...")
    # split_data_filename = os.path.join(MODEL_DIR, f"train_test_split_data_{START_DATE}_to_{END_DATE}.npz")
    np.savez_compressed(SPLIT_DATA_FILENAME,
                          X_train=X_train_np,
                          X_test=X_test_np,
                          y_train=y_train,
                          y_test=y_test,
                          feature_names=final_feature_names) # Also save feature names
    print(f"✅ Split data saved to {SPLIT_DATA_FILENAME}")

--- Scaling (Fit on Train, Transform Train & Test) ---

In [ ]:
if not data_is_loaded_and_split:
    print("\nScaling numerical features...")
    # Identify indices of numerical features in the NumPy array
    # Use the list of column names generated by get_dummies
    numerical_indices = [i for i, col_name in enumerate(final_feature_names) if col_name in numerical_features]
    # numerical_indices = [final_feature_names.index(col) for col in numerical_features if col in final_feature_names] # Alternative
    if not numerical_indices:
         print("⚠️ No numerical features found after encoding. Check feature lists.")
    else:
        print(f"Indices of numerical features to scale: {numerical_indices[:5]}... (Total: {len(numerical_indices)})")
    
        scaler = StandardScaler()
    
        # Fit scaler ONLY on the training data's numerical features
        scaler.fit(X_train_np[:, numerical_indices])
    
        # Transform both training and test sets
        # Ensure arrays are float type before in-place modification
        X_train_scaled_np = X_train_np.astype(float, copy=True)
        X_test_scaled_np = X_test_np.astype(float, copy=True)
    
        X_train_scaled_np[:, numerical_indices] = scaler.transform(X_train_np[:, numerical_indices])
        X_test_scaled_np[:, numerical_indices] = scaler.transform(X_test_np[:, numerical_indices])
    
        print("Scaling complete. Sample of scaled training data (first 5 rows, first 5 numerical features):")
        print(X_train_scaled_np[:5, numerical_indices]) # Print a small sample of scaled numerical cols
else: 
    print("Scaling data exists assigning to x vars") 
    # Create final X_train, X_test as copies to apply scaling
    X_train = X_train_np.astype(np.float32, copy=True)
    X_test = X_test_np.astype(np.float32, copy=True)

In [ ]:
if not data_is_loaded_and_split:
    # --- Save Scaler ---
    save_scaler_start_time = time.time()
    print(f"\n💾 Saving scaler to {SCALER_FILENAME}")
    os.makedirs(os.path.dirname(SCALER_FILENAME), exist_ok=True)
    with open(SCALER_FILENAME, 'wb') as f:
        pickle.dump(scaler, f)
    print("✅ Scaler saved.")
    print(f"Saving scaler took: {time.time() - save_scaler_start_time:.2f}s")
    
    print(f"⏱️ Total Preprocessing Time (excluding load): {time.time() - section_start_time:.2f} seconds")

In [ ]:
#%% [markdown]
# ### Verify Loaded/Split y_test Distribution

#%%
print("\n--- Checking y_test distribution before RF evaluation ---")

# Check if y_test exists and is a NumPy array
if 'y_test' in locals() and isinstance(y_test, np.ndarray):
    unique_test, counts_test = np.unique(y_test, return_counts=True)
    print("y_test unique values and counts:")
    print(np.asarray((unique_test, counts_test))) # Prints [[unique_vals], [counts]]
    
    # Explicitly check for presence of class 1
    if 1 not in unique_test:
        print("\n---> CONFIRMED: y_test contains only class 0! <---")
    elif len(unique_test) == 2:
         positive_class_index = np.where(unique_test == 1)[0][0]
         print(f"\ny_test contains {counts_test[positive_class_index]} positive samples (class 1).")
    else:
         print("\ny_test contains unexpected unique values.")

else:
    print("❌ ERROR: 'y_test' is not defined or is not a NumPy array before evaluation!")

### Model Experiments (Random Forest & LightGBM)


In [ ]:
def evaluate_model(name, model, X_test, y_test):
    eval_start_time = time.time()
    print(f"\n--- Evaluating {name} ---")
    # Check if input arrays are valid
    if X_test is None or y_test is None:
        print("❌ Cannot evaluate model: X_test or y_test is None.")
        return {'pr_auc': -1, 'f1': -1}
        
    y_pred = model.predict(X_test)
    y_proba = None # Initialize probability variable
    pr_auc = -1    # Default PR AUC if probabilities aren't available/valid
    f1 = f1_score(y_test, y_pred, zero_division=0) # F1 based on thresholded predictions
    
    # Try to get probabilities
    if hasattr(model, "predict_proba"):
        try:
            probabilities = model.predict_proba(X_test)
            if probabilities.shape[1] == 2:
                y_proba = probabilities[:, 1] # Get probability of positive class (class 1)
                pr_auc = average_precision_score(y_test, y_proba)
                print(f"Average Precision (PR AUC): {pr_auc:.4f}")
            elif probabilities.shape[1] == 1:
                print("⚠️ Warning: predict_proba returned only one column. Model likely predicted only one class.")
                print("   Cannot reliably calculate PR AUC or plot PR curve based on P(class=1).")
                # You could try to infer based on model.classes_ if needed, but safest is to report F1 only.
                pr_auc = -1 # Indicate PR AUC is not applicable/calculable here
            else:
                print(f"⚠️ Warning: predict_proba returned unexpected shape: {probabilities.shape}. Cannot calculate PR AUC.")
                pr_auc = -1

        except Exception as proba_err:
            print(f"❌ Error getting or processing probabilities: {proba_err}")
            pr_auc = -1
            y_proba = None # Ensure y_proba is None if error occurs
    else:
        print("Model does not have predict_proba. Cannot calculate PR AUC.")
        pr_auc = -1 # Indicate PR AUC is not applicable

    # Print metrics that don't rely on probabilities
    print("Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(f"F1 Score (Positive Class): {f1:.4f}") # F1 score is based on y_pred

    # Plot PR curve only if probabilities for class 1 were successfully obtained
    if y_proba is not None and pr_auc > -1:
        try:
            precision, recall, _ = precision_recall_curve(y_test, y_proba)
            pr_display_auc = auc(recall, precision)
            plt.figure(figsize=(8, 6))
            plt.plot(recall, precision, label=f'{name} (AUC = {pr_display_auc:.3f})')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.title(f'{name} Precision-Recall curve')
            plt.legend(loc='best')
            plt.grid(True)
            plt.show() # Ensure plot is displayed
        except Exception as plot_err:
            print(f"⚠️ Could not generate PR curve plot: {plot_err}")
    elif pr_auc == -1:
         print("(PR Curve skipped as probabilities for class 1 were not available or valid)")


    print(f"⏱️ Evaluation for {name} took: {time.time() - eval_start_time:.2f} seconds")
    # Return metrics, PR AUC might be -1 if calculation wasn't possible
    return {'pr_auc': pr_auc, 'f1': f1}

--- Random Forest ---

In [ ]:
model_start_time = time.time()
print("\nTraining Random Forest...")
# Add try-except block for robustness
try:
    if 'X_train' in locals() and isinstance(X_train, np.ndarray):
        rf = RandomForestClassifier(class_weight='balanced',
                                    random_state=42, n_jobs=-1,
                                    n_estimators=150, max_depth=20,
                                    min_samples_leaf=10, min_samples_split=20)
        rf.fit(X_train, y_train)
        print(f"⏱️ Random Forest Training took: {time.time() - model_start_time:.2f} seconds")
        results['Random Forest'] = evaluate_model('Random Forest', rf, X_test, y_test)
    else:
         print("❌ X_train (scaled NumPy array) not found. Cannot train Random Forest.")
         results['Random Forest'] = {'pr_auc': -1, 'f1': -1}
except Exception as rf_err:
    print(f"❌ Error training Random Forest: {rf_err}")
    results['Random Forest'] = {'pr_auc': -1, 'f1': -1}